In [1]:
import os

In [2]:
%pwd

'e:\\Replica\\wine_mlops\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'e:\\Replica\\wine_mlops'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    """Data transformation configuration"""
    root_dir: Path
    data_path: Path

In [7]:
from wine_quality.constants import *
from wine_quality.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    """Configuration manager to manage all configurations"""
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path)
        )

        return data_transformation_config

In [9]:
import os
from wine_quality import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [10]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

        # You can add different transformation techniques like scaling, encoding, etc. here
        # We can also perfomr different kinds of EDA before passing it to ML model
        # For now, we will just split the data into train and test sets, as the data is already clean

    def train_test_splitting(self):
        """Splitting the data into train and test sets"""
        logger.info("Splitting the data into train and test sets")

        # Read the data
        data = pd.read_csv(self.config.data_path)

        # Split the data into train and test sets
        train, test = train_test_split(data, test_size=0.2, random_state=42)

        # Save the train and test sets to the root directory
        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info("Data split into train and test sets")
        logger.info(f"Train set saved to {os.path.join(self.config.root_dir, 'train.csv')}")
        logger.info(f"Test set saved to {os.path.join(self.config.root_dir, 'test.csv')}")
        logger.info(train.shape)
        logger.info(test.shape)

        print(train.shape)
        print(test.shape)


In [11]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_splitting()

except Exception as e:
    raise e

[2025-05-01 07:30:14,238: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-05-01 07:30:14,240: INFO: common: yaml file: params.yaml loaded successfully]
[2025-05-01 07:30:14,242: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-05-01 07:30:14,244: INFO: common: created directory at: artifacts]
[2025-05-01 07:30:14,245: INFO: common: created directory at: artifacts/data_transformation]
[2025-05-01 07:30:14,245: INFO: 2983377752: Splitting the data into train and test sets]
[2025-05-01 07:30:14,273: INFO: 2983377752: Data split into train and test sets]
[2025-05-01 07:30:14,274: INFO: 2983377752: Train set saved to artifacts\data_transformation\train.csv]
[2025-05-01 07:30:14,275: INFO: 2983377752: Test set saved to artifacts\data_transformation\test.csv]
[2025-05-01 07:30:14,275: INFO: 2983377752: (1279, 12)]
[2025-05-01 07:30:14,275: INFO: 2983377752: (320, 12)]
(1279, 12)
(320, 12)
